In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, udf, explode, round
from pyspark.sql.types import DoubleType

In [0]:
spark = SparkSession.builder.appName("StudentAverageScore").getOrCreate()

# Sample nested data
data = [
    (1, "John", [{"subject": "Math", "score": 85}, {"subject": "science", "score": 90}]),
    (2, "Jane", [{"subject": "Math", "score": 77}, {"subject": "science", "score": 86}]),
    (3, "Emily", [{"subject": "Math", "score": 97}, {"subject": "science", "score": 93}])
]

In [0]:
df = spark.createDataFrame(data, ["id", "name", "scores"])
df.show(truncate=False)


+---+-----+-------------------------------------------------------------------+
|id |name |scores                                                             |
+---+-----+-------------------------------------------------------------------+
|1  |John |[{subject -> Math, score -> 85}, {subject -> science, score -> 90}]|
|2  |Jane |[{subject -> Math, score -> 77}, {subject -> science, score -> 86}]|
|3  |Emily|[{subject -> Math, score -> 97}, {subject -> science, score -> 93}]|
+---+-----+-------------------------------------------------------------------+



In [0]:
exploded_df = df.withColumn("score", explode(col('scores')))
exploded_df.show(truncate=False)

+---+-----+-------------------------------------------------------------------+---------------------------------+
|id |name |scores                                                             |score                            |
+---+-----+-------------------------------------------------------------------+---------------------------------+
|1  |John |[{subject -> Math, score -> 85}, {subject -> science, score -> 90}]|{subject -> Math, score -> 85}   |
|1  |John |[{subject -> Math, score -> 85}, {subject -> science, score -> 90}]|{subject -> science, score -> 90}|
|2  |Jane |[{subject -> Math, score -> 77}, {subject -> science, score -> 86}]|{subject -> Math, score -> 77}   |
|2  |Jane |[{subject -> Math, score -> 77}, {subject -> science, score -> 86}]|{subject -> science, score -> 86}|
|3  |Emily|[{subject -> Math, score -> 97}, {subject -> science, score -> 93}]|{subject -> Math, score -> 97}   |
|3  |Emily|[{subject -> Math, score -> 97}, {subject -> science, score -> 93}]|{subject 

In [0]:
pivot_df = exploded_df.select('id', 'name', 'score.subject', 'score.score')\
    .groupBy("id", "name")\
    .pivot("subject")\
    .agg(expr("first(score)"))
pivot_df.show(truncate=False)

+---+-----+----+-------+
|id |name |Math|science|
+---+-----+----+-------+
|1  |John |85  |90     |
|2  |Jane |77  |86     |
|3  |Emily|97  |93     |
+---+-----+----+-------+



In [0]:
# Define UDF to calculate average
def compute_avg(*scores):
    valid_scores = [float(score) for score in scores if score is not None]
    return sum(valid_scores) / len(valid_scores) if valid_scores else None

# Register UDF
avg_udf = udf(compute_avg, DoubleType())
# Apply UDF to calculate average scores
result_df = pivot_df.withColumn("average_score", avg_udf(col('Math'), col('science')))
result_df.show(truncate=False)

+---+-----+----+-------+-------------+
|id |name |Math|science|average_score|
+---+-----+----+-------+-------------+
|1  |John |85  |90     |87.5         |
|2  |Jane |77  |86     |81.5         |
|3  |Emily|97  |93     |95.0         |
+---+-----+----+-------+-------------+



In [0]:
# Average scores without UDF
subjects = [c for c in pivot_df.columns if c not in ('id', 'name')]
values = [col(sub) for sub in subjects]
final_df2 = pivot_df.withColumn('avg_score', round(sum(values)/len(values), 2))
final_df2.show(truncate=False)

+---+-----+----+-------+---------+
|id |name |Math|science|avg_score|
+---+-----+----+-------+---------+
|1  |John |85  |90     |87.5     |
|2  |Jane |77  |86     |81.5     |
|3  |Emily|97  |93     |95.0     |
+---+-----+----+-------+---------+

